# Kvasir-VQA x1 — Text-only baselines

Train text-only classifiers on questions:
- Yes/No binary classification
- Open-ended top-K answer classification

Outputs saved to `2_modeling/01_text_only/out/`.


In [1]:
from pathlib import Path
import json
import random
import re

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [2]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "01_text_only" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
TOP_K = 200
MAX_FEATURES = 5000
NGRAM_RANGE = (1, 2)

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)


Data root: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/arcturus/Desktop/old/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/01_text_only/out


In [3]:
# Load metadata
meta = pd.read_csv(META_CSV)

# Normalize text fields
meta["question_norm"] = meta["question"].fillna("").astype(str).str.lower().str.strip()
meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 46966, 'val': 5931, 'test': 5952}


In [4]:
# Utilities

def fit_eval_text_classifier(train_df, val_df, test_df, label_col, out_prefix):
    vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=NGRAM_RANGE)
    clf = LogisticRegression(max_iter=1000, n_jobs=-1)

    X_train = vectorizer.fit_transform(train_df["question_norm"])
    y_train = train_df[label_col].values
    clf.fit(X_train, y_train)

    def eval_split(df, split_name):
        X = vectorizer.transform(df["question_norm"])
        y_true = df[label_col].values
        y_pred = clf.predict(X)
        metrics = {
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro"))
        }
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        pred_df = df[["question", "answer", label_col]].copy()
        pred_df["pred"] = y_pred
        pred_df.to_csv(OUT_DIR / f"{out_prefix}_pred_{split_name}.csv", index=False)
        with open(OUT_DIR / f"{out_prefix}_metrics_{split_name}.json", "w") as f:
            json.dump({"metrics": metrics, "report": report}, f, indent=2)
        print(split_name, metrics)

    eval_split(train_df, "train")
    if len(val_df):
        eval_split(val_df, "val")
    if len(test_df):
        eval_split(test_df, "test")

    return clf, vectorizer


In [5]:
# Yes/No classification (question only)

def normalize_yesno(text):
    t = re.sub(r"[^a-z]", "", str(text).lower())
    if t in ("yes", "no"):
        return t
    return None

meta["answer_yesno"] = meta["answer"].apply(normalize_yesno)
yn_df = meta[meta["answer_yesno"].notna()].reset_index(drop=True)

train_yn = yn_df[yn_df["split"] == "train"].reset_index(drop=True)
val_yn = yn_df[yn_df["split"] == "validation"].reset_index(drop=True)
test_yn = yn_df[yn_df["split"] == "test"].reset_index(drop=True)

print("Yes/No rows:", len(yn_df))
if len(train_yn):
    fit_eval_text_classifier(train_yn, val_yn, test_yn, "answer_yesno", "yesno_text")


Yes/No rows: 15243
train {'accuracy': 0.7814270465555464, 'macro_f1': 0.7809319237152077}
val {'accuracy': 0.7782152230971129, 'macro_f1': 0.7776023929096925}
test {'accuracy': 0.7779220779220779, 'macro_f1': 0.7772906486428887}


In [6]:
# Top-K answer classification (question only)

# Build top-K answers from train split only to avoid leakage
answer_counts = train_df["answer_norm"].value_counts()
TOP_K_ANSWERS = answer_counts.head(TOP_K).index.tolist()

train_k = train_df[train_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
val_k = val_df[val_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
test_k = test_df[test_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)

print("Top-K answers:", len(TOP_K_ANSWERS))
print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})

if len(train_k):
    fit_eval_text_classifier(train_k, val_k, test_k, "answer_norm", "topk_text")


Top-K answers: 200
{'train': 46598, 'val': 5884, 'test': 5893}
train {'accuracy': 0.6537405038842868, 'macro_f1': 0.04086006780379597}
val {'accuracy': 0.6526172671651937, 'macro_f1': 0.05755593440380406}
test {'accuracy': 0.6526387239097234, 'macro_f1': 0.05818740242360812}
